# MuscleMap Thigh Segmentation — Augmented Dataset (Lambda)

Runs **MuscleMap thigh** segmentation on the 20 augmented NIfTI water volumes.
Input is already NIfTI — no DICOM conversion needed.

MuscleMap requires Python 3.11; a conda env is created automatically.

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
Output: `~/musclemap_thigh_augmented_segs/{stem}_augmented000_water_dseg.nii.gz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_thigh_augmented_segs/ \
  /path/to/local/muscle_map_thigh/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os, shutil, glob
import numpy as np
import SimpleITK as sitk

_conda_candidates = [
    shutil.which('conda'),
    os.path.expanduser('~/miniconda3/bin/conda'),
    os.path.expanduser('~/anaconda3/bin/conda'),
    '/opt/conda/bin/conda',
]
CONDA = next((p for p in _conda_candidates if p and os.path.exists(p)), None)
if CONDA is None:
    subprocess.check_call(['bash', '-c',
        'wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh '
        '-O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -p ~/miniconda3'])
    CONDA = os.path.expanduser('~/miniconda3/bin/conda')

ENV_NAME = 'musclemap_env'
ENV_DIR  = os.path.join(os.path.dirname(os.path.dirname(CONDA)), 'envs', ENV_NAME)
ENV_PY   = os.path.join(ENV_DIR, 'bin', 'python')
MM_BIN   = os.path.join(ENV_DIR, 'bin', 'mm_segment')

if not os.path.exists(ENV_PY):
    subprocess.check_call([CONDA, 'create', '-n', ENV_NAME, 'python=3.11', 'pip',
                           '-c', 'conda-forge', '--override-channels', '-y', '-q'])
subprocess.check_call([ENV_PY, '-m', 'pip', 'install', '-q',
                       'git+https://github.com/MuscleMap/MuscleMap.git'])
print('mm_segment exists:', os.path.exists(MM_BIN))

In [ ]:
DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
OUTPUT_DIR = os.path.expanduser('~/musclemap_thigh_augmented_segs')

os.makedirs(OUTPUT_DIR, exist_ok=True)

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Found {len(nii_files)} NIfTI water volumes')

run_env = os.environ.copy()
run_env.pop('MPLBACKEND', None)

for nii_path in nii_files:
    basename = os.path.basename(nii_path)            # e.g. HV001_1_stack1_augmented000_water.nii.gz
    stem     = basename.replace('_water.nii.gz', '')  # e.g. HV001_1_stack1_augmented000_water
    # mm_segment names its output {input_stem}_dseg.nii.gz
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dseg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    subprocess.check_call([
        MM_BIN, '-i', nii_path, '-r', 'thigh', '-o', OUTPUT_DIR, '-g', 'Y',
    ], env=run_env)

    if not os.path.exists(out_path):
        # Search for the output in case mm_segment used a different naming convention
        candidates = glob.glob(os.path.join(OUTPUT_DIR, f'{stem}*dseg*.nii.gz'))
        if candidates:
            shutil.move(candidates[0], out_path)
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_dseg.nii.gz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(results[0]))
    print(f'Sample shape: {arr.shape}  labels: {sorted(np.unique(arr[arr>0]).tolist())[:10]}')